# Arrow IPC - Rust

All 12 Rust examples from [docs/ipc.md](https://platob.github.io/yggdryl/ipc/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

## Arrow batch reads and writes

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
let schema = field.into_arrow_schema()?;
let batch = |ids: Vec<i64>, venues: Vec<Option<&str>>| {
    RecordBatch::try_new(
        Arc::clone(&schema),
        vec![
            Arc::new(Int64Array::from(ids)),
            Arc::new(StringArray::from(venues)),
        ],
    )
};

let mut handle = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
let options = handle.record_options()?;
handle.overwrite_arrow_reader(
    yggdryl::arrow::batch_reader(
        Arc::clone(&schema),
        [batch(vec![1, 2], vec![Some("XNAS"), Some("XNYS")])?],
    ),
    &options,
)?;
handle.append_arrow_reader(
    yggdryl::arrow::batch_reader(
        Arc::clone(&schema),
        [batch(vec![3], vec![Some("XLON")])?],
    ),
    &options,
)?;
handle.merge_arrow_reader(
    yggdryl::arrow::batch_reader(
        Arc::clone(&schema),
        [batch(vec![2, 4], vec![Some("XPAR"), None])?],
    ),
    &options.clone().with_merge_by_names(["id"]),
)?;

let rows = handle
    .read_arrow_reader(&options)?
    .map(|batch| batch.map(|batch| batch.num_rows()))
    .sum::<Result<usize, _>>()?;
assert_eq!(rows, 4);

### Dimensions and opened sessions

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::generic::Holder;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let schema = field.clone().into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&schema),
    vec![Arc::new(Int64Array::from(vec![1, 2]))],
)?;
let mut handle = Holder::buffer(Buffer::new().with_media_type(MimeType::ARROW_STREAM.into()));
let options = handle.record_options()?;
handle.overwrite_arrow_reader(yggdryl::arrow::batch_reader(schema, [batch]), &options)?;

handle.open()?;
assert_eq!(handle.read_arrow_field(&options)?, field);
assert_eq!((handle.row_size()?, handle.column_size()?), (2, 1));
handle.close()?;

## Reading and writing are both readers

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, RecordBatchReader};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::ipc::{self, IpcOptions};
use yggdryl::{DataType, MimeType};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = schema.into_arrow_schema()?;
let batches = (0..3)
    .map(|start| {
        RecordBatch::try_new(
            arrow_schema.clone(),
            vec![Arc::new(Int64Array::from(vec![start, start + 1]))],
        )
    })
    .collect::<Result<Vec<_>, _>>()?;

let mut handle = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
let options = IpcOptions::new();

// `batch_reader` turns whatever is already in hand - a Vec, an array, an
// iterator - into the one shape a write takes.
ipc::overwrite_batch_reader(
    &mut handle,
    arrow::batch_reader(arrow_schema, batches),
    &options,
)?;

let reader = ipc::read_batch_reader(&handle, None, &options)?;
// The schema is known before a single batch is decoded.
assert_eq!(reader.schema().fields().len(), 1);

let mut rows = 0;
for batch in reader {
    rows += batch?.num_rows();
}
assert_eq!(rows, 6);

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::ipc::{self, IpcOptions};
use yggdryl::{DataType, MimeType};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = schema.into_arrow_schema()?;

let mut handle = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());

// Nothing is materialized: each batch is built as the writer asks for it.
let produced = (0..4).map({
    let arrow_schema = arrow_schema.clone();
    move |start| {
        RecordBatch::try_new(
            arrow_schema.clone(),
            vec![Arc::new(Int64Array::from(vec![start]))],
        )
        .expect("batch")
    }
});
ipc::overwrite_batch_reader(
    &mut handle,
    arrow::batch_reader(arrow_schema, produced),
    &IpcOptions::new(),
)?;

assert_eq!(
    ipc::read_batch_reader(&handle, None, &IpcOptions::new())?.count(),
    4
);

## Column pushdown

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, RecordBatchReader, StringArray};
use yggdryl::arrow;
use yggdryl::io::Buffer;
use yggdryl::ipc::{self, IpcOptions};
use yggdryl::{DataType, MimeType};

let stored = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.required_field("symbol"),
    DataType::Utf8.required_field("venue"),
])?
.required_field("row");
let arrow_schema = stored.into_arrow_schema()?;

let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(vec![1, 2])),
        Arc::new(StringArray::from(vec!["AAPL", "MSFT"])),
        Arc::new(StringArray::from(vec!["XNAS", "XNAS"])),
    ],
)?;

let mut handle = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
let options = IpcOptions::new();
ipc::overwrite_batch_reader(&mut handle, arrow::batch_reader(arrow_schema, [batch]), &options)?;

// One of the three columns, named by a root Field of its own.
let wanted = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

let projected = ipc::read_batch_reader(&handle, Some(&wanted), &options)?;
assert_eq!(projected.schema().fields().len(), 1);
let batches = projected.collect::<Result<Vec<_>, _>>()?;
assert_eq!(batches[0].num_columns(), 1);

// The stream itself is unchanged: it still carries all three.
assert_eq!(ipc::read_field(&handle, &options)?.field_len(), 3);

## One stream, one configuration

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::ipc::Ipc;
use yggdryl::{DataType, Url};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = schema.clone().into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from(vec![1, 2]))],
)?;

let handle = Buffer::new().with_media_type(Url::from_str("file:///trades.arrows")?.media_type());
let mut media = Ipc::new(handle).with_field(schema.clone());
let options = media.record_options()?;

// One options value carries the schema, root name, and coding.
media.overwrite_arrow_reader(arrow::batch_reader(arrow_schema, [batch]), &options)?;
assert_eq!(media.read_arrow_reader(&options)?.count(), 1);
assert_eq!(media.read_arrow_field(&options)?, schema);

// An Ipc is also the bytes it encodes: a stream opens with its continuation marker.
assert_eq!(media.read_range(0, 4)?, [0xFF, 0xFF, 0xFF, 0xFF]);

## The stream carries its schema

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::ipc::{Ipc, DEFAULT_ROOT_NAME};
use yggdryl::DataType;

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = schema.clone().into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from(vec![7]))],
)?;

let mut writer = Ipc::new(Buffer::new()).with_field(schema.clone());
let options = writer.record_options()?;
writer.overwrite_arrow_reader(arrow::batch_reader(arrow_schema, [batch]), &options)?;
let bytes = writer.handle().as_slice().to_vec();

// A reader that declares nothing recovers the schema from the bytes.
let reader = Ipc::new(Buffer::from_bytes(bytes.clone()));
let options = reader.record_options()?;
assert_eq!(reader.read_arrow_field(&options)?, schema);
assert_eq!(reader.read_arrow_field(&options)?.name(), DEFAULT_ROOT_NAME);

// Arrow names columns, not the record; the root name is chosen on this side.
let named = Ipc::new(Buffer::from_bytes(bytes)).with_root_name("trade");
let options = named.record_options()?;
let named_field = named.read_arrow_field(&options)?;
assert_eq!(named_field.name(), "trade");
assert_eq!(named_field.get_field_by_name("id"), schema.get_field_by_name("id"));

## Content coding comes from the name

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::ipc::Ipc;
use yggdryl::{DataType, Url};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = schema.clone().into_arrow_schema()?;

let mut sizes = Vec::new();
for name in ["trades.arrows", "trades.arrows.gz", "trades.arrows.zst"] {
    let url = Url::from_str(&format!("file:///{name}"))?;
    let handle = Buffer::new().with_media_type(url.media_type());
    let mut media = Ipc::new(handle).with_field(schema.clone());

    let batch = RecordBatch::try_new(
        arrow_schema.clone(),
        vec![Arc::new(Int64Array::from(vec![1, 2]))],
    )?;
    let options = media.record_options()?;
    media.overwrite_arrow_reader(
        arrow::batch_reader(arrow_schema.clone(), [batch]),
        &options,
    )?;

    // Identical calls on both sides, whatever the coding is.
    assert_eq!(media.read_arrow_reader(&options)?.count(), 1, "{name}");
    sizes.push(media.handle().as_slice().to_vec());
}

// The bytes underneath are framed by the coding the name declared.
assert_eq!(&sizes[1][..2], &[0x1F, 0x8B]);
assert_eq!(&sizes[2][..4], &[0x28, 0xB5, 0x2F, 0xFD]);
assert_ne!(sizes[0], sizes[1]);

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::ipc::Ipc;
use yggdryl::{DataType, Level, Url};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = schema.clone().into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from((0..512).collect::<Vec<i64>>()))],
)?;

let handle = Buffer::new().with_media_type(Url::from_str("file:///trades.arrows.gz")?.media_type());
let mut media = Ipc::new(handle)
    .with_field(schema.clone())
    .with_level(Level::BEST);
let options = media.record_options()?;

media.overwrite_arrow_reader(
    yggdryl::arrow::batch_reader(arrow_schema, [batch]),
    &options,
)?;
assert_eq!(media.read_arrow_reader(&options)?.count(), 1);
// Still a gzip member, and smaller than the stream it encodes.
assert_eq!(&media.handle().as_slice()[..2], &[0x1F, 0x8B]);
assert!(media.handle().size() < 512 * 8);

## Options

In [ ]:
use yggdryl::generic::{IORecordOptions, RecordOptions};
use yggdryl::ipc::{IpcOptions, DEFAULT_ROOT_NAME};
use yggdryl::{DataType, Level, MimeType};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

let options = IpcOptions::new()
    .with_field(schema.clone())
    .with_level(Level::BEST);

assert_eq!(options.field(), Some(&schema));
assert_eq!(options.root_name(), DEFAULT_ROOT_NAME);
assert_eq!(options.level(), Level::BEST);

// The fields are public, so a setting can also be written directly.
let mut direct = IpcOptions::new();
direct.batch_size = Some(1024);
assert_eq!(direct.batch_size(), Some(1024));

// It converts into the enum every encoding's settings share.
let erased: RecordOptions = options.into();
assert_eq!(erased.mime_type(), MimeType::ARROW_STREAM);

## Absence

In [ ]:
use arrow_array::{RecordBatch, RecordBatchReader};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::ipc::Ipc;
use yggdryl::DataType;

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

// A resource that does not exist yet holds no batches; it is not a parse failure.
let missing = Ipc::new(Buffer::new()).with_field(schema.clone());
let options = missing.record_options()?;
let reader = missing.read_arrow_reader(&options)?;
// The declared schema is what the empty reader reports.
assert_eq!(reader.schema().fields().len(), 1);
assert_eq!(reader.count(), 0);

// Opening an absent stream succeeds and caches explicit zero dimensions.
let mut empty = Ipc::new(Buffer::new());
empty.open()?;
assert!(empty.opened());
assert_eq!(empty.row_size()?, 0);
assert_eq!(empty.column_size()?, 0);

// Writing no batches still writes the schema, so the stream exists and is readable.
let mut written = Ipc::new(Buffer::new()).with_field(schema.clone());
let options = written.record_options()?;
written.overwrite_arrow_reader(
    arrow::batch_reader(
        schema.clone().into_arrow_schema()?,
        std::iter::empty::<RecordBatch>(),
    ),
    &options,
)?;
assert!(!written.handle().is_empty());
assert_eq!(written.read_arrow_reader(&options)?.count(), 0);
assert_eq!(written.read_arrow_field(&options)?, schema);

In [ ]:
use yggdryl::io::Buffer;
use yggdryl::ipc::{self, IpcOptions};

let handle = Buffer::from_bytes(b"definitely not an Arrow IPC stream".to_vec());
assert!(ipc::read_field(&handle, &IpcOptions::new()).is_err());
assert!(ipc::read_batch_reader(&handle, None, &IpcOptions::new()).is_err());